# NumPy for Data Analytics — Complete Notes

| Part | Topics |
|---|---|
| **1. Fundamentals** | Import, loading/saving data, missing values, creating arrays, attributes, data types |
| **2. Indexing & Slicing** | Basic & negative indexing, slicing with steps, boolean masks, `np.where`, fancy indexing, views vs copies |
| **3. Data Transformation** | Reshape, transpose, combine/split, sort, unique, clip/round, maths functions, broadcasting, normalisation, `diff`, filling missing values |
| **4. Statistical Analysis** | Aggregates & `axis`, `argmin`/`argmax`, `ptp`, std/var, weighted average, percentiles & quantiles, IQR & outliers, cumulative functions, correlation & covariance, histograms, NaN-safe functions |

**Dataset:** `numpy_video1_sales_dataset.csv`: monthly sales for one year (12 rows × 6 columns + header).

| Column | Index | Meaning |
|---|---|---|
| Month | 0 | Month number 1–12 |
| Electronics | 1 | Sales |
| Clothing | 2 | Sales |
| Groceries | 3 | Sales |
| Furniture | 4 | Sales |
| Toys | 5 | Sales |

Keep the CSV **in the same folder as this notebook**, then Run All.

# Part 1 — NumPy Fundamentals

## 1.1 Import

`np` is the standard alias. `np.set_printoptions(suppress=True)` stops scientific notation (`1.25e+04`) so figures stay readable.

In [100]:
import numpy as np
import os

np.set_printoptions(suppress=True)
print("NumPy version:", np.__version__)

NumPy version: 2.5.4


## 1.2 Check the file is visible

| Function | Purpose |
|---|---|
| `os.getcwd()` | Folder the notebook is running in |
| `os.listdir()` | Files in that folder |
| `os.path.exists(path)` | `True` if the file exists |

A plain file name is looked up in the current folder. `FileNotFoundError` means the CSV isn't there.

In [101]:
print("Current folder:", os.getcwd())
print("CSV found?     ", os.path.exists('numpy_video1_sales_dataset.csv'))

Current folder: /Users/jkj/Desktop/intellea/Intellea_Python_classes
CSV found?      True


In [102]:
# Google Colab only: upload the file first
# from google.colab import files
# uploaded = files.upload()

## 1.3 Loading data: `np.genfromtxt()`

```python
data = np.genfromtxt('numpy_video1_sales_dataset.csv', delimiter=',', skip_header=1)
```

| Parameter | What it does |
|---|---|
| `fname` | File path |
| `delimiter=','` | Separator between values |
| `skip_header=N` | Skip the first N lines |
| `skip_footer=N` | Skip the last N lines |
| `max_rows=N` | Read only N rows |
| `usecols=(0, 2)` | Read only some columns |
| `dtype` | Type to read as (default `float`) |
| `names=True` | Use header as column names (structured array) |
| `missing_values` / `filling_values` | Strings to treat as missing / value to put in their place |
| `encoding='utf-8'` | Text encoding |

Values that can't be read become `nan` (Not a Number) instead of raising an error.

In [103]:
data = np.genfromtxt('numpy_video1_sales_dataset.csv', delimiter=',', skip_header=1)
print(data)

[[    1. 12500.  8300. 15200.  6400.  3100.]
 [    2. 11800.  7900. 14800.  5900.  2800.]
 [    3. 13200.  9100. 15900.  7100.  3300.]
 [    4. 12900.  8700. 16100.  6800.  3500.]
 [    5. 14100.  9400. 16500.  7300.  3700.]
 [    6. 13800.  9900. 17200.  7000.  3600.]
 [    7. 15200. 10200. 17800.  7600.  4100.]
 [    8. 14900.  9800. 17400.  7400.  3900.]
 [    9. 13600.  9300. 16800.  6900.  3800.]
 [   10. 16100. 10800. 18300.  8200.  4500.]
 [   11. 18400. 12600. 19100.  9100.  5900.]
 [   12. 21300. 14900. 20800. 10400.  7800.]]


In [104]:
# Header as text: read only line 1
columns = np.genfromtxt('numpy_video1_sales_dataset.csv', delimiter=',', max_rows=1, dtype=str)
print(columns)

['Month' 'Electronics' 'Clothing' 'Groceries' 'Furniture' 'Toys']


In [105]:
# Integers instead of floats
data_int = np.genfromtxt('numpy_video1_sales_dataset.csv', delimiter=',', skip_header=1, dtype=int)
print(data_int[:2])

[[    1 12500  8300 15200  6400  3100]
 [    2 11800  7900 14800  5900  2800]]


In [106]:
# Selected columns
month_elec = np.genfromtxt('numpy_video1_sales_dataset.csv', delimiter=',', skip_header=1, usecols=(0, 1))
print(month_elec.shape)     # (12, 2)

toys_only = np.genfromtxt('numpy_video1_sales_dataset.csv', delimiter=',', skip_header=1, usecols=5)
print(toys_only.shape)      # (12,) → a single column gives 1-D

(12, 2)
(12,)


In [107]:
# Selected rows
print(np.genfromtxt('numpy_video1_sales_dataset.csv', delimiter=',', skip_header=1, max_rows=3).shape)     # (3, 6) Jan–Mar
print(np.genfromtxt('numpy_video1_sales_dataset.csv', delimiter=',', skip_header=1, skip_footer=6).shape)  # (6, 6) Jan–Jun

(3, 6)
(6, 6)


In [108]:
# names=True → access columns by name
named = np.genfromtxt('numpy_video1_sales_dataset.csv', delimiter=',', names=True)
print(named.dtype.names)
print(named['Toys'])
print(named.shape)   # (12,) → 12 records, not a 2-D table

('Month', 'Electronics', 'Clothing', 'Groceries', 'Furniture', 'Toys')
[3100. 2800. 3300. 3500. 3700. 3600. 4100. 3900. 3800. 4500. 5900. 7800.]
(12,)


## 1.4 Missing values when loading

`io.StringIO` creates a small CSV in memory, which `genfromtxt` reads like a file. Empty cells and text become `nan`.

| Tool | Purpose |
|---|---|
| `np.isnan(arr)` | `True` where a value is `nan` |
| `.sum()` on that | Count of missing values |
| `filling_values=0` | Replace missing values while loading |

In [109]:
import io

csv_text = """Month,Electronics,Clothing
1,12500,8300
2,,7900
3,13200,abc"""

m = np.genfromtxt(io.StringIO(csv_text), delimiter=',', skip_header=1)
print(m)
print(np.isnan(m))
print("Missing:", np.isnan(m).sum())

filled = np.genfromtxt(io.StringIO(csv_text), delimiter=',', skip_header=1, filling_values=0)
print(filled)

[[    1. 12500.  8300.]
 [    2.    nan  7900.]
 [    3. 13200.    nan]]
[[False False False]
 [False  True False]
 [False False  True]]
Missing: 2
[[    1. 12500.  8300.]
 [    2.     0.  7900.]
 [    3. 13200.     0.]]


In [110]:
# Note: nan is never equal to anything, even itself, so use np.isnan, not ==
print(np.nan == np.nan)         # False
print(np.isnan(np.nan))         # True
print("Real data missing:", np.isnan(data).sum())

False
True
Real data missing: 0


## 1.5 `np.loadtxt()`: strict loader

| | `genfromtxt` | `loadtxt` |
|---|---|---|
| Missing/bad values | become `nan` | error |
| Skip header | `skip_header` | `skiprows` |
| Speed | slower | faster |

In [111]:
data_lt = np.loadtxt('numpy_video1_sales_dataset.csv', delimiter=',', skiprows=1)
print(np.array_equal(data, data_lt))   # True

True


## 1.6 Saving: `np.savetxt()` and `np.save()`

| Function | Format | Notes |
|---|---|---|
| `np.savetxt(f, arr, delimiter=',', fmt='%d', header=..., comments='')` | CSV/text | readable; `fmt='%.2f'` for 2 decimals |
| `np.save('x.npy', arr)` / `np.load('x.npy')` | NumPy binary | fast, keeps dtype and shape exactly |

In [112]:
np.savetxt('q1_sales.csv', data_int[:3], delimiter=',', fmt='%d',
           header=','.join(columns), comments='')
print(open('q1_sales.csv').read())

np.save('sales.npy', data)
print(np.array_equal(np.load('sales.npy'), data))   # True

Month,Electronics,Clothing,Groceries,Furniture,Toys
1,12500,8300,15200,6400,3100
2,11800,7900,14800,5900,2800
3,13200,9100,15900,7100,3300

True


## 1.7 Creating arrays

| Function | Creates | Example |
|---|---|---|
| `np.array(list)` | From a Python list | `np.array([1, 2, 3])` |
| `np.zeros(shape)` | All 0 | `np.zeros((2, 3))` |
| `np.ones(shape)` | All 1 | `np.ones(4)` |
| `np.full(shape, value)` | All the same value | `np.full((2, 2), 7)` |
| `np.arange(start, stop, step)` | Evenly stepped values (stop excluded) | `np.arange(1, 13)` |
| `np.linspace(start, stop, n)` | n evenly spaced values (stop included) | `np.linspace(0, 1, 5)` |
| `np.eye(n)` | Identity matrix | `np.eye(3)` |
| `np.zeros_like(arr)` | Same shape as `arr`, all 0 | `np.zeros_like(data)` |
| `rng.integers / rng.normal` | Random values | see below |

In [113]:
print(np.array([[1, 2], [3, 4]]))
print(np.zeros((2, 3)))
print(np.full((2, 2), 7))
print(np.arange(1, 13))           # month numbers 1–12
print(np.arange(0, 1, 0.25))      # float steps
print(np.linspace(0, 1, 5))       # [0, 0.25, 0.5, 0.75, 1]
print(np.eye(3))
print(np.zeros_like(data).shape)  # (12, 6)

[[1 2]
 [3 4]]
[[0. 0. 0.]
 [0. 0. 0.]]
[[7 7]
 [7 7]]
[ 1  2  3  4  5  6  7  8  9 10 11 12]
[0.   0.25 0.5  0.75]
[0.   0.25 0.5  0.75 1.  ]
[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]
(12, 6)


In [114]:
# Random numbers: create a generator; a seed makes results repeatable
rng = np.random.default_rng(seed=42)
print(rng.integers(1, 100, size=5))          # 5 random ints from 1–99
print(rng.normal(loc=0, scale=1, size=3))    # normal distribution (mean 0, std 1)
print(rng.choice(columns[1:], size=2, replace=False))   # random pick

[ 9 77 65 44 43]
[ 0.94056472 -1.95103519 -1.30217951]
['Furniture' 'Toys']


## 1.8 Array attributes

Attributes are written **without brackets**.

| Attribute | Meaning | Our data |
|---|---|---|
| `ndim` | Number of dimensions | `2` |
| `shape` | Size of each dimension | `(12, 6)` |
| `size` | Total elements | `72` |
| `len(arr)` | Length of first dimension | `12` |
| `dtype` | Element type | `float64` |
| `itemsize` | Bytes per element | `8` |
| `nbytes` | Total bytes (`size × itemsize`) | `576` |
| `type(arr)` | Object type | `numpy.ndarray` |

In [115]:
print("ndim:    ", data.ndim)
print("shape:   ", data.shape, "| rows:", data.shape[0], "| cols:", data.shape[1])
print("size:    ", data.size, "=", np.prod(data.shape))
print("len:     ", len(data))
print("dtype:   ", data.dtype)
print("itemsize:", data.itemsize)
print("nbytes:  ", data.nbytes)
print("type:    ", type(data))

ndim:     2
shape:    (12, 6) | rows: 12 | cols: 6
size:     72 = 72
len:      12
dtype:    float64
itemsize: 8
nbytes:   576
type:     <class 'numpy.ndarray'>


In [116]:
# ndim and shape for other dimensions
a1 = np.array([1, 2, 3])
a3 = np.zeros((2, 3, 4))      # 2 tables of 3×4
print(a1.ndim, a1.shape)      # 1 (3,)  → comma means 1-element tuple
print(a3.ndim, a3.shape)      # 3 (2, 3, 4)

1 (3,)
3 (2, 3, 4)


## 1.9 Data types and `astype()`

An array holds **one type** for all elements. Mixing types upcasts everything to the most general one (int → float → str).

| dtype | Meaning | Bytes |
|---|---|---|
| `int32`, `int64` | Whole numbers | 4, 8 |
| `float32`, `float64` | Decimals | 4, 8 |
| `bool` | True/False | 1 |
| `<U11` | Text, up to 11 characters | – |

`astype()` returns a **new** array with the new type; the original is unchanged. Converting float to int **truncates** (drops decimals, no rounding).

In [117]:
print(np.array([1, 2.5]).dtype)        # float64 → int upcast to float
print(np.array([1, 'a']).dtype)        # <U21 → everything becomes text
print(columns.dtype)                   # <U11

small = data.astype(np.float32)
print(small.dtype, small.nbytes)       # float32 288 → half the memory

print(np.array([2.9, -2.9]).astype(int))   # [ 2 -2] → truncated, not rounded

float64
<U21
<U11
float32 288
[ 2 -2]


# Part 2 — Array Indexing & Slicing

## 2.1 Basic indexing

Indexes start at **0**. Negative indexes count from the end (`-1` = last).

| Syntax | Gets |
|---|---|
| `arr[r, c]` | One element |
| `arr[r]` | Whole row |
| `arr[:, c]` | Whole column |
| `arr[-1]` | Last row |
| `arr[r][c]` | Same as `arr[r, c]` but slower; prefer the comma form |

In [118]:
print(data[0, 1])     # Jan Electronics → 12500.0
print(data[-1])       # December row
print(data[:, 1])     # Electronics column
print(data[-1, -1])   # December Toys

12500.0
[   12. 21300. 14900. 20800. 10400.  7800.]
[12500. 11800. 13200. 12900. 14100. 13800. 15200. 14900. 13600. 16100.
 18400. 21300.]
7800.0


## 2.2 Slicing: `start:stop:step`

`stop` is **excluded**. Any part can be left out: `:3` = from start, `3:` = to end, `::2` = every 2nd.

| Syntax | Gets |
|---|---|
| `arr[0:3]` | Rows 0, 1, 2 |
| `arr[6:]` | Row 6 to end |
| `arr[:, 1:]` | All rows, columns 1 to end |
| `arr[::2]` | Every 2nd row |
| `arr[::-1]` | Reversed |
| `arr[0:3, 2:4]` | Block of rows and columns |

In [119]:
print(data[0:3].shape)        # (3, 6) Jan–Mar
print(data[6:].shape)         # (6, 6) Jul–Dec
print(data[::3, 0])           # [1, 4, 7, 10] → every 3rd month
print(data[::-1, 0])          # months reversed
print(data[0:3, 2:4])         # Jan–Mar, Clothing & Groceries

(3, 6)
(6, 6)
[ 1.  4.  7. 10.]
[12. 11. 10.  9.  8.  7.  6.  5.  4.  3.  2.  1.]
[[ 8300. 15200.]
 [ 7900. 14800.]
 [ 9100. 15900.]]


In [120]:
# Handy names used through the rest of the notebook
months     = data[:, 0]
sales      = data[:, 1:]        # (12, 5) without Month
categories = columns[1:]
electronics = sales[:, 0]
print(sales.shape, categories)

(12, 5) ['Electronics' 'Clothing' 'Groceries' 'Furniture' 'Toys']


## 2.3 Boolean indexing (masks)

A comparison gives an array of `True`/`False` (a **mask**). Putting it inside `[]` keeps only the `True` positions.

| Operator | Meaning |
|---|---|
| `>`, `<`, `>=`, `<=`, `==`, `!=` | Comparisons |
| `&` | AND |
| `\|` | OR |
| `~` | NOT |

Use `&`, `|`, `~` (not `and`, `or`, `not`), and wrap each condition in brackets.

In [121]:
mask = electronics > 15000
print(mask)
print(electronics[mask])          # values above 15000
print(months[mask])               # which months
print(mask.sum())                 # how many → True counts as 1

[False False False False False False  True False False  True  True  True]
[15200. 16100. 18400. 21300.]
[ 7. 10. 11. 12.]
4


In [122]:
# Combined conditions
both = (sales[:, 0] > 14000) & (sales[:, 1] > 10000)   # Electronics AND Clothing high
print(months[both])

either = (sales[:, 4] > 5000) | (sales[:, 3] > 9000)   # Toys OR Furniture high
print(months[either])

print(months[~mask])              # NOT above 15000

[ 7. 10. 11. 12.]
[11. 12.]
[1. 2. 3. 4. 5. 6. 8. 9.]


In [123]:
# Filter whole rows: all data for months where Groceries > 17000
print(data[data[:, 3] > 17000])

[[    6. 13800.  9900. 17200.  7000.  3600.]
 [    7. 15200. 10200. 17800.  7600.  4100.]
 [    8. 14900.  9800. 17400.  7400.  3900.]
 [   10. 16100. 10800. 18300.  8200.  4500.]
 [   11. 18400. 12600. 19100.  9100.  5900.]
 [   12. 21300. 14900. 20800. 10400.  7800.]]


## 2.4 `np.where()`

| Form | Returns |
|---|---|
| `np.where(condition)` | Indexes where condition is True |
| `np.where(condition, a, b)` | `a` where True, `b` where False (like if/else for every element) |

In [124]:
idx = np.where(electronics > 15000)
print(idx)                  # tuple of index arrays
print(months[idx])

labels = np.where(electronics > 15000, 'High', 'Normal')
print(labels)

# Nested: three levels
level = np.where(electronics > 18000, 'Peak', np.where(electronics > 13000, 'Good', 'Low'))
print(level)

(array([ 6,  9, 10, 11]),)
[ 7. 10. 11. 12.]
['Normal' 'Normal' 'Normal' 'Normal' 'Normal' 'Normal' 'High' 'Normal'
 'Normal' 'High' 'High' 'High']
['Low' 'Low' 'Good' 'Low' 'Good' 'Good' 'Good' 'Good' 'Good' 'Good' 'Peak'
 'Peak']


## 2.5 Fancy indexing (lists of indexes)

Pass a list or array of indexes to pick specific rows or columns in any order.

In [125]:
print(data[[0, 5, 11]])         # Jan, Jun, Dec
print(data[:, [0, 3, 5]][:3])   # Month, Groceries, Toys (first 3 rows)
print(categories[[4, 0]])       # reorder: Toys, Electronics

[[    1. 12500.  8300. 15200.  6400.  3100.]
 [    6. 13800.  9900. 17200.  7000.  3600.]
 [   12. 21300. 14900. 20800. 10400.  7800.]]
[[    1. 15200.  3100.]
 [    2. 14800.  2800.]
 [    3. 15900.  3300.]]
['Toys' 'Electronics']


## 2.6 Finding positions: `np.nonzero`, `np.argwhere`, `np.isin`

| Function | Returns |
|---|---|
| `np.nonzero(mask)` | Indexes of True values (same as `np.where(mask)`) |
| `np.argwhere(mask)` | `[row, col]` pairs of True values |
| `np.isin(arr, values)` | True where elements are in `values` |

In [126]:
print(np.nonzero(electronics > 18000))
print(np.argwhere(sales > 18000))           # [row, col] of every value above 18000

festive = np.isin(months, [10, 11, 12])     # Oct–Dec
print(sales[festive].sum())

(array([10, 11]),)
[[ 9  2]
 [10  0]
 [10  2]
 [11  0]
 [11  2]]
198200.0


## 2.7 Modifying with indexing

Assign to an index, slice or mask to change values in place.

In [127]:
d = data.copy()                 # work on a copy (see 2.8)
d[0, 1] = 0                     # one value
d[:, 5] = d[:, 5] * 1.1         # whole column +10%
d[d[:, 4] < 6500, 4] = 6500     # set Furniture floor at 6500
print(d[:3])

[[    1.     0.  8300. 15200.  6500.  3410.]
 [    2. 11800.  7900. 14800.  6500.  3080.]
 [    3. 13200.  9100. 15900.  7100.  3630.]]


## 2.8 Views vs copies

| | Created by | Changing it changes the original? |
|---|---|---|
| **View** | Basic slicing `arr[0:3]`, `reshape`, `.T`, `ravel` | **Yes** |
| **Copy** | `.copy()`, boolean/fancy indexing, `flatten` | No |

Use `.copy()` when you want to edit a slice safely. `np.shares_memory(a, b)` checks.

In [128]:
view = data[0:2]
copy = data[0:2].copy()
print(np.shares_memory(view, data))   # True
print(np.shares_memory(copy, data))   # False

tmp = data.copy()
v = tmp[0:2]
v[0, 0] = 999
print(tmp[0, 0])                       # 999.0 → original changed through the view

True
False
999.0


# Part 3 — Data Transformation

## 3.1 Changing shape

Size must stay the same (72).

| Tool | Does | View or copy |
|---|---|---|
| `reshape(r, c)` | New shape; `-1` = work it out | view |
| `ravel()` | Flatten to 1-D | view (if possible) |
| `flatten()` | Flatten to 1-D | copy |
| `.T` / `transpose()` | Swap rows and columns | view |
| `np.newaxis` / `np.expand_dims` | Add a dimension of size 1 | view |
| `squeeze()` | Remove dimensions of size 1 | view |

In [129]:
print(data.flatten().shape)          # (72,)
print(data.reshape(6, 12).shape)     # (6, 12)
print(data.reshape(-1, 3).shape)     # (24, 3)

# 3-D: 4 quarters × 3 months × 6 columns
quarters = data.reshape(4, 3, 6)
print(quarters.ndim, quarters.shape)
print(quarters[3])                   # Q4 (Oct–Dec)

print(data.T.shape)                  # (6, 12)

(72,)
(6, 12)
(24, 3)
3 (4, 3, 6)
[[   10. 16100. 10800. 18300.  8200.  4500.]
 [   11. 18400. 12600. 19100.  9100.  5900.]
 [   12. 21300. 14900. 20800. 10400.  7800.]]
(6, 12)


In [130]:
col = electronics[:, np.newaxis]     # (12,) → (12, 1) column vector
print(col.shape)
print(np.expand_dims(electronics, axis=0).shape)   # (1, 12) row vector
print(col.squeeze().shape)           # back to (12,)

try:
    data.reshape(10, 7)              # 70 ≠ 72
except ValueError as e:
    print("Error:", e)

(12, 1)
(1, 12)
(12,)
Error: cannot reshape array of size 72 into shape (10,7)


## 3.2 Combining arrays

| Function | Joins |
|---|---|
| `np.concatenate([a, b], axis=0)` | Along rows (stack vertically) |
| `np.concatenate([a, b], axis=1)` | Along columns |
| `np.vstack([a, b])` | Vertically (adds rows) |
| `np.hstack([a, b])` | Horizontally (adds columns) |
| `np.column_stack([a, b])` | 1-D arrays become columns |

Shapes must match along the other axis.

In [131]:
h1, h2 = data[:6], data[6:]
print(np.vstack([h1, h2]).shape)                    # (12, 6)
print(np.concatenate([h1, h2], axis=0).shape)       # (12, 6)

total = sales.sum(axis=1)
with_total = np.column_stack([data, total])         # add a Total column
print(with_total.shape)                             # (12, 7)
print(np.hstack([data, total[:, np.newaxis]]).shape)  # same with hstack

(12, 6)
(12, 6)
(12, 7)
(12, 7)


## 3.3 Splitting arrays

| Function | Splits |
|---|---|
| `np.split(arr, n)` | Into n equal parts along rows |
| `np.split(arr, [i, j])` | At given indexes |
| `np.vsplit` / `np.hsplit` | By rows / by columns |

In [132]:
q1, q2, q3, q4 = np.split(data, 4)        # 4 quarters of 3 rows
print(q4[:, 0])

month_col, sales_part = np.hsplit(data, [1])   # split columns at index 1
print(month_col.shape, sales_part.shape)       # (12, 1) (12, 5)

[10. 11. 12.]
(12, 1) (12, 5)


## 3.4 Adding and removing: `np.append`, `np.insert`, `np.delete`

All return a **new** array (slow in loops; collect in a list then convert instead).

In [133]:
new_row = np.array([[13, 20000, 13000, 19000, 9000, 6000]])
print(np.append(data, new_row, axis=0).shape)          # (13, 6)
print(np.delete(data, 0, axis=1).shape)                # drop Month column → (12, 5)
print(np.insert(categories, 0, 'Month'))               # insert at position 0

(13, 6)
(12, 5)
['Month' 'Electronics' 'Clothing' 'Groceries' 'Furniture' 'Toys']


## 3.5 Sorting: `np.sort` and `argsort`

| Function | Returns |
|---|---|
| `np.sort(arr)` | Sorted copy (ascending) |
| `arr.sort()` | Sorts in place |
| `np.argsort(arr)` | Indexes that would sort the array → use to sort other arrays the same way |
| `[::-1]` | Reverse for descending |

In [134]:
print(np.sort(electronics))                 # ascending
print(np.sort(electronics)[::-1])           # descending

order = np.argsort(electronics)[::-1]       # months by Electronics, best first
print(months[order].astype(int))

[11800. 12500. 12900. 13200. 13600. 13800. 14100. 14900. 15200. 16100.
 18400. 21300.]
[21300. 18400. 16100. 15200. 14900. 14100. 13800. 13600. 13200. 12900.
 12500. 11800.]
[12 11 10  7  8  5  6  9  3  4  1  2]


In [135]:
# Sort whole rows by a column (Groceries, highest first)
by_groc = data[data[:, 3].argsort()[::-1]]
print(by_groc[:3])

# Rank categories by yearly total
totals = sales.sum(axis=0)
print(categories[np.argsort(totals)[::-1]])

# sort along an axis: each column sorted independently
print(np.sort(sales, axis=0)[:2])

[[   12. 21300. 14900. 20800. 10400.  7800.]
 [   11. 18400. 12600. 19100.  9100.  5900.]
 [   10. 16100. 10800. 18300.  8200.  4500.]]
['Groceries' 'Electronics' 'Clothing' 'Furniture' 'Toys']
[[11800.  7900. 14800.  5900.  2800.]
 [12500.  8300. 15200.  6400.  3100.]]


## 3.6 Unique values: `np.unique`

| Option | Adds |
|---|---|
| `return_counts=True` | How often each value appears |
| `return_index=True` | First position of each value |

In [136]:
labels = np.where(electronics > 15000, 'High', 'Normal')
vals, counts = np.unique(labels, return_counts=True)
print(vals, counts)

print(np.unique(np.array([3, 1, 3, 2, 1])))     # sorted unique → [1 2 3]

['High' 'Normal'] [4 8]
[1 2 3]


## 3.7 Rounding, clipping and maths functions

These work **element-wise** (on every value at once, no loop needed).

| Function | Does |
|---|---|
| `np.round(x, n)` | Round to n decimals |
| `np.floor` / `np.ceil` | Round down / up |
| `np.clip(x, lo, hi)` | Limit values to a range |
| `np.abs` | Absolute value |
| `np.sqrt`, `np.square`, `np.power(x, p)` | Roots and powers |
| `np.log`, `np.log10`, `np.exp` | Logs and exponential |
| `np.maximum(a, b)` / `np.minimum(a, b)` | Element-wise larger / smaller of two arrays (≠ `max`) |

In [137]:
x = electronics / 1000
print(np.round(x, 1))
print(np.floor(x), np.ceil(x))
print(np.clip(electronics, 13000, 16000))     # cap low/high values
print(np.sqrt(electronics[:3]))
print(np.log10(electronics[:3]).round(3))
print(np.maximum(sales[:, 0], sales[:, 2]))   # bigger of Electronics vs Groceries each month

[12.5 11.8 13.2 12.9 14.1 13.8 15.2 14.9 13.6 16.1 18.4 21.3]
[12. 11. 13. 12. 14. 13. 15. 14. 13. 16. 18. 21.] [13. 12. 14. 13. 15. 14. 16. 15. 14. 17. 19. 22.]
[13000. 13000. 13200. 13000. 14100. 13800. 15200. 14900. 13600. 16000.
 16000. 16000.]
[111.80339887 108.62780491 114.89125293]
[4.097 4.072 4.121]
[15200. 14800. 15900. 16100. 16500. 17200. 17800. 17400. 16800. 18300.
 19100. 21300.]


## 3.8 Vectorised arithmetic and broadcasting

Arithmetic between arrays happens element by element. **Broadcasting** lets arrays of different shapes work together by stretching the smaller one, if trailing dimensions are equal or one of them is 1.

| Shapes | Works? |
|---|---|
| `(12, 5)` with a number | ✔ |
| `(12, 5)` with `(5,)` | ✔ one value per column |
| `(12, 5)` with `(12, 1)` | ✔ one value per row |
| `(12, 5)` with `(12,)` | ✘ use `[:, np.newaxis]` or `keepdims=True` |

`keepdims=True` keeps the collapsed axis as size 1, so the result broadcasts back.

In [138]:
print((sales * 1.18)[:2])                 # add 18% GST to everything

target = np.array([15000, 10000, 17000, 8000, 4000])   # one target per category
print((sales - target)[:2])               # (12,5) − (5,) → per column

monthly = sales.sum(axis=1, keepdims=True)    # (12, 1)
share = sales / monthly * 100                 # % share of each category in each month
print(share.round(1)[:2])
print(share.sum(axis=1)[:2])                  # each row adds to 100

[[14750.  9794. 17936.  7552.  3658.]
 [13924.  9322. 17464.  6962.  3304.]]
[[-2500. -1700. -1800. -1600.  -900.]
 [-3200. -2100. -2200. -2100. -1200.]]
[[27.5 18.2 33.4 14.1  6.8]
 [27.3 18.3 34.3 13.7  6.5]]
[100. 100.]


## 3.9 Normalisation and scaling

| Method | Formula | Result |
|---|---|---|
| Min-max | `(x − min) / (max − min)` | 0 to 1 |
| Z-score | `(x − mean) / std` | mean 0, std 1 |
| % of total | `x / total × 100` | percentages |

Use `axis=0` to scale each column separately.

In [139]:
mn, mx = sales.min(axis=0), sales.max(axis=0)
minmax = (sales - mn) / (mx - mn)
print(minmax.round(2)[:3])

z = (sales - sales.mean(axis=0)) / sales.std(axis=0)
print(z.round(2)[:3])
print(z.mean(axis=0).round(6), z.std(axis=0))   # ~0 and 1

[[0.07 0.06 0.07 0.11 0.06]
 [0.   0.   0.   0.   0.  ]
 [0.15 0.17 0.18 0.27 0.1 ]]
[[-0.89 -0.95 -1.21 -0.94 -0.8 ]
 [-1.17 -1.16 -1.46 -1.37 -1.03]
 [-0.62 -0.52 -0.78 -0.35 -0.65]]
[ 0.  0.  0.  0. -0.] [1. 1. 1. 1. 1.]


## 3.10 Changes over time: `np.diff` and percentage change

`np.diff(arr)` gives each value minus the previous one, so the result is 1 shorter.

In [140]:
totals_m = sales.sum(axis=1)
change = np.diff(totals_m)
print(change)                                   # month-over-month change

pct_change = np.diff(totals_m) / totals_m[:-1] * 100
print(pct_change.round(1))

print(months[1:][change < 0])                   # months where sales dropped

[-2300.  5400.  -600.  3000.   500.  3400. -1500. -3000.  7500.  7200.
 10100.]
[-5.1 12.5 -1.2  6.2  1.   6.6 -2.7 -5.6 14.9 12.4 15.5]
[2. 4. 8. 9.]


## 3.11 Filling and cleaning missing values

| Tool | Does |
|---|---|
| `np.nan_to_num(arr, nan=0)` | Replace `nan` with a number |
| `arr[np.isnan(arr)] = value` | Replace using a mask |
| `np.nanmean(arr, axis=0)` | Mean ignoring `nan` (use to fill with column mean) |
| `arr[~np.isnan(arr).any(axis=1)]` | Drop rows containing any `nan` |

In [141]:
messy = data[:4, 1:3].copy()
messy[1, 0] = np.nan
messy[3, 1] = np.nan
print(messy)

print(np.nan_to_num(messy, nan=0))

col_means = np.nanmean(messy, axis=0)
filled = np.where(np.isnan(messy), col_means, messy)   # broadcast column means
print(filled)

print(messy[~np.isnan(messy).any(axis=1)])   # rows with no missing values

[[12500.  8300.]
 [   nan  7900.]
 [13200.  9100.]
 [12900.    nan]]
[[12500.  8300.]
 [    0.  7900.]
 [13200.  9100.]
 [12900.     0.]]
[[12500.          8300.        ]
 [12866.66666667  7900.        ]
 [13200.          9100.        ]
 [12900.          8433.33333333]]
[[12500.  8300.]
 [13200.  9100.]]


# Part 4 — Statistical Analysis

## 4.1 Aggregate functions

| Function | Returns |
|---|---|
| `sum()` | Total |
| `mean()` | Average |
| `np.median()` | Middle value |
| `min()` / `max()` | Smallest / largest value |
| `argmin()` / `argmax()` | Index of smallest / largest |
| `np.ptp()` | Range = max − min |
| `std()` / `var()` | Standard deviation / variance |
| `prod()` | Product |
| `np.count_nonzero()` | Number of non-zero / True values |

Most work as `arr.func()` or `np.func(arr)`; `median` and `ptp` are used as `np.func()`.

In [142]:
print("Sum:   ", sales.sum())
print("Mean:  ", sales.mean())
print("Median:", np.median(sales))
print("Min:   ", sales.min(), "| Max:", sales.max())
print("Range: ", np.ptp(sales))
print("Std:   ", sales.std().round(2))
print("Var:   ", sales.var().round(2))
print("Count > 15000:", np.count_nonzero(sales > 15000))

Sum:    644700.0
Mean:   10745.0
Median: 9850.0
Min:    2800.0 | Max: 21300.0
Range:  18500.0
Std:    5057.55
Var:    25578808.33
Count > 15000: 15


## 4.2 The `axis` argument

| `axis` | Collapses | Result for `sales` (12×5) |
|---|---|---|
| none | everything | one number |
| `0` | down the rows | one per **column** (category) → `(5,)` |
| `1` | across columns | one per **row** (month) → `(12,)` |

Memory trick: the axis you name is the one that **disappears**.

In [143]:
cat_total   = sales.sum(axis=0)
month_total = sales.sum(axis=1)
print(categories)
print(cat_total)
print(month_total)

print("Mean per category: ", sales.mean(axis=0).round(1))
print("Median per category:", np.median(sales, axis=0))
print("Range per category:", np.ptp(sales, axis=0))

['Electronics' 'Clothing' 'Groceries' 'Furniture' 'Toys']
[177800. 120900. 205900.  90100.  50000.]
[45500. 43200. 48600. 48000. 51000. 51500. 54900. 53400. 50400. 57900.
 65100. 75200.]
Mean per category:  [14816.7 10075.  17158.3  7508.3  4166.7]
Median per category: [13950.  9600. 17000.  7200.  3750.]
Range per category: [9500. 7000. 6000. 4500. 5000.]


In [144]:
# 3-D: total per quarter per column
print(quarters.sum(axis=1)[:, 1:])     # (4, 5) → collapses the 3 months

[[37500. 25300. 45900. 19400.  9200.]
 [40800. 28000. 49800. 21100. 10800.]
 [43700. 29300. 52000. 21900. 11800.]
 [55800. 38300. 58200. 27700. 18200.]]


## 4.3 `argmin()` and `argmax()`

They return the **index**, not the value. Use it to look up a month or category.

On a 2-D array without `axis`, the index counts as if the array were flattened; `np.unravel_index` converts it back to `(row, col)`.

In [145]:
print("Best month: ", int(months[month_total.argmax()]))
print("Worst month:", int(months[month_total.argmin()]))
print("Best category: ", categories[cat_total.argmax()])
print("Worst category:", categories[cat_total.argmin()])

Best month:  12
Worst month: 2
Best category:  Groceries
Worst category: Toys


In [146]:
# Per column: which month was lowest for each category
for cat, i in zip(categories, sales.argmin(axis=0)):
    print(f"{cat:12} lowest in month {int(months[i])}")

# Per row: top category each month
print(categories[sales.argmax(axis=1)])

Electronics  lowest in month 2
Clothing     lowest in month 2
Groceries    lowest in month 2
Furniture    lowest in month 2
Toys         lowest in month 2
['Groceries' 'Groceries' 'Groceries' 'Groceries' 'Groceries' 'Groceries'
 'Groceries' 'Groceries' 'Groceries' 'Groceries' 'Groceries' 'Electronics']


In [147]:
flat_i = sales.argmax()
r, c = np.unravel_index(flat_i, sales.shape)
print(flat_i, "→ row", r, "col", c, "→", categories[c], "in month", int(months[r]))

55 → row 11 col 0 → Electronics in month 12


## 4.4 Spread: standard deviation, variance, `ddof`

- **Variance** = average squared distance from the mean.
- **Standard deviation** = √variance, in the same units as the data.
- `ddof=0` (default) = population; `ddof=1` = sample (divides by n−1). Pandas uses `ddof=1` by default, so results can differ.
- **Coefficient of variation** = std / mean: compares spread between columns of different size.

In [148]:
print(sales.std(axis=0).round(1))           # population
print(sales.std(axis=0, ddof=1).round(1))   # sample
print(np.sqrt(sales.var(axis=0)).round(1))  # std = sqrt(var)

cv = sales.std(axis=0) / sales.mean(axis=0) * 100
print("CV %:", cv.round(1))                 # Toys is the most variable relative to its size

[2589.3 1872.7 1619.4 1174.3 1331.2]
[2704.5 1955.9 1691.4 1226.6 1390.4]
[2589.3 1872.7 1619.4 1174.3 1331.2]
CV %: [17.5 18.6  9.4 15.6 31.9]


## 4.5 Weighted average: `np.average`

`np.mean` treats every value equally; `np.average(arr, weights=w)` gives some values more importance.

In [149]:
# Average Electronics sale, giving festive months (Oct–Dec) double weight
w = np.where(months >= 10, 2, 1)
print("Plain mean:   ", electronics.mean().round(1))
print("Weighted mean:", np.average(electronics, weights=w).round(1))

# Average monthly sale across categories, weighted by each category's yearly total
print(np.average(sales.mean(axis=0), weights=cat_total).round(1))

Plain mean:    14816.7
Weighted mean: 15573.3
12828.0


## 4.6 Percentiles and quantiles

The **p-th percentile** is the value below which p% of the data falls.

| Call | Meaning |
|---|---|
| `np.percentile(arr, 50)` | Median |
| `np.percentile(arr, [25, 50, 75])` | Quartiles Q1, Q2, Q3 |
| `np.percentile(arr, 90, axis=0)` | Per column |
| `np.quantile(arr, 0.25)` | Same, but 0–1 scale |
| `method='linear'` (default), `'lower'`, `'higher'`, `'nearest'`, `'midpoint'` | How to pick a value between two data points |

In [150]:
print(np.percentile(electronics, 50), np.median(electronics))   # equal
q1_e, q2_e, q3_e = np.percentile(electronics, [25, 50, 75])
print("Q1:", q1_e, "| Q2:", q2_e, "| Q3:", q3_e)
print(np.quantile(electronics, 0.75))                           # = Q3

pcts = np.percentile(sales, [25, 50, 75], axis=0)
print(pcts)     # row 0 = Q1, row 1 = median, row 2 = Q3 for each category

13950.0 13950.0
Q1: 13125.0 | Q2: 13950.0 | Q3: 15425.0
15425.0
[[13125.  9000. 16050.  6875.  3450.]
 [13950.  9600. 17000.  7200.  3750.]
 [15425. 10350. 17925.  7750.  4200.]]


In [151]:
for m in ['linear', 'lower', 'higher', 'nearest', 'midpoint']:
    print(f"{m:9}", np.percentile(electronics, 25, method=m))

linear    13125.0
lower     12900.0
higher    13200.0
nearest   13200.0
midpoint  13050.0


In [152]:
p90 = np.percentile(month_total, 90)
print("Top 10% months:", months[month_total >= p90].astype(int))

Top 10% months: [11 12]


## 4.7 IQR and outlier detection

- **IQR** = Q3 − Q1 (spread of the middle 50%).
- Common outlier rule: below `Q1 − 1.5×IQR` or above `Q3 + 1.5×IQR`.
- Z-score rule: `|z| > 2` (or 3).

In [153]:
q1_s, q3_s = np.percentile(sales, [25, 75], axis=0)
iqr = q3_s - q1_s
lower, upper = q1_s - 1.5 * iqr, q3_s + 1.5 * iqr
print("IQR:  ", iqr)
print("Upper:", upper)

outliers = (sales < lower) | (sales > upper)     # broadcasting per column
for r, c in np.argwhere(outliers):
    print(f"Outlier: {categories[c]} in month {int(months[r])} = {sales[r, c]}")

IQR:   [2300. 1350. 1875.  875.  750.]
Upper: [18875.  12375.  20737.5  9062.5  5325. ]
Outlier: Clothing in month 11 = 12600.0
Outlier: Furniture in month 11 = 9100.0
Outlier: Toys in month 11 = 5900.0
Outlier: Electronics in month 12 = 21300.0
Outlier: Clothing in month 12 = 14900.0
Outlier: Groceries in month 12 = 20800.0
Outlier: Furniture in month 12 = 10400.0
Outlier: Toys in month 12 = 7800.0


In [154]:
z = (sales - sales.mean(axis=0)) / sales.std(axis=0)
print(np.argwhere(np.abs(z) > 2))     # [row, col] with |z| > 2

[[11  0]
 [11  1]
 [11  2]
 [11  3]
 [11  4]]


## 4.8 Cumulative functions

| Function | Returns |
|---|---|
| `cumsum()` | Running total |
| `cumprod()` | Running product (e.g. compound growth) |
| `np.maximum.accumulate()` | Running maximum (best so far) |

In [155]:
running = month_total.cumsum()
print(running)
print("Month total passes 300k:", int(months[(running > 300000).argmax()]))

growth = np.array([1.05, 1.10, 0.95, 1.08])
print(growth.cumprod().round(3))      # compounded growth factor

print(np.maximum.accumulate(electronics))   # highest Electronics sale so far

[ 45500.  88700. 137300. 185300. 236300. 287800. 342700. 396100. 446500.
 504400. 569500. 644700.]
Month total passes 300k: 7
[1.05  1.155 1.097 1.185]
[12500. 12500. 13200. 13200. 14100. 14100. 15200. 15200. 15200. 16100.
 18400. 21300.]


## 4.9 Correlation and covariance

| Function | Returns |
|---|---|
| `np.corrcoef(a, b)` | Correlation matrix; value from −1 to 1 |
| `np.cov(a, b)` | Covariance matrix (direction, not scaled) |

Rows are treated as variables, so pass `rowvar=False` (or use `.T`) when variables are columns.

| r | Meaning |
|---|---|
| ≈ 1 | Move together |
| ≈ 0 | No linear relation |
| ≈ −1 | Move opposite |

In [156]:
r = np.corrcoef(sales[:, 0], sales[:, 4])
print(r)                  # 2×2 matrix; r[0, 1] is the correlation
print("Electronics vs Toys:", r[0, 1].round(3))

corr = np.corrcoef(sales, rowvar=False)    # all categories at once (5×5)
print(categories)
print(corr.round(2))

[[1.         0.98578224]
 [0.98578224 1.        ]]
Electronics vs Toys: 0.986
['Electronics' 'Clothing' 'Groceries' 'Furniture' 'Toys']
[[1.   0.99 0.97 0.99 0.99]
 [0.99 1.   0.97 0.99 0.99]
 [0.97 0.97 1.   0.96 0.94]
 [0.99 0.99 0.96 1.   0.98]
 [0.99 0.99 0.94 0.98 1.  ]]


In [157]:
print(np.cov(sales[:, 0], sales[:, 4]).round(0))
# Correlation with time: are sales trending up?
print("Electronics vs month:", np.corrcoef(months, electronics)[0, 1].round(3))

[[7314242. 3706970.]
 [3706970. 1933333.]]
Electronics vs month: 0.857


## 4.10 Frequency distribution: `np.histogram`, `np.bincount`, `np.digitize`

| Function | Does |
|---|---|
| `np.histogram(arr, bins=n)` | Counts per bin + bin edges |
| `np.digitize(arr, edges)` | Which bin each value falls in |
| `np.bincount(int_arr)` | Counts of each non-negative integer |

In [158]:
counts, edges = np.histogram(month_total, bins=3)
print(counts)
print(edges.round(0))

bands = np.digitize(month_total, [50000, 60000])   # 0: <50k, 1: 50–60k, 2: ≥60k
print(bands)
print(np.bincount(bands))                          # months in each band

[8 2 2]
[43200. 53867. 64533. 75200.]
[0 0 0 0 1 1 1 1 1 1 2 2]
[4 6 2]


## 4.11 NaN-safe statistics

Normal functions return `nan` if any value is missing. The `nan` versions ignore missing values.

| Normal | NaN-safe |
|---|---|
| `sum`, `mean`, `median` | `np.nansum`, `np.nanmean`, `np.nanmedian` |
| `min`, `max`, `argmin`, `argmax` | `np.nanmin`, `np.nanmax`, `np.nanargmin`, `np.nanargmax` |
| `std`, `var` | `np.nanstd`, `np.nanvar` |
| `percentile`, `quantile` | `np.nanpercentile`, `np.nanquantile` |
| `cumsum` | `np.nancumsum` |

In [159]:
e = electronics.copy()
e[2] = np.nan
print(e.mean())                    # nan
print(np.nanmean(e).round(1))
print(np.nanmax(e), np.nanargmax(e))
print(np.nanpercentile(e, 75))

nan
14963.6
21300.0 11
15650.0


## 4.12 Logical summaries: `any`, `all`

| Function | True if |
|---|---|
| `np.any(cond)` | At least one is True |
| `np.all(cond)` | Every one is True |

In [160]:
print(np.any(sales > 20000))              # any value over 20k?
print(np.all(sales > 2000))               # every value over 2k?
print(np.all(np.diff(running) > 0))       # running total always increasing?
print((sales > 10000).all(axis=1).astype(int))   # months where every category > 10k → none

True
True
True
[0 0 0 0 0 0 0 0 0 0 0 0]


## 4.13 Putting it together: a sales report

In [161]:
print("=== Annual Sales Report ===")
print(f"Total sales: {sales.sum():,.0f}")
print(f"Best month:  {int(months[month_total.argmax()])} ({month_total.max():,.0f})")
print(f"Worst month: {int(months[month_total.argmin()])} ({month_total.min():,.0f})")
print()
print(f"{'Category':12}{'Total':>10}{'Share%':>8}{'Mean':>9}{'Std':>8}{'Growth%':>9}")
share_cat = cat_total / cat_total.sum() * 100
growth_cat = (sales[-1] - sales[0]) / sales[0] * 100
for i, cat in enumerate(categories):
    print(f"{cat:12}{cat_total[i]:>10,.0f}{share_cat[i]:>8.1f}"
          f"{sales[:, i].mean():>9,.0f}{sales[:, i].std():>8,.0f}{growth_cat[i]:>9.1f}")

=== Annual Sales Report ===
Total sales: 644,700
Best month:  12 (75,200)
Worst month: 2 (43,200)

Category         Total  Share%     Mean     Std  Growth%
Electronics    177,800    27.6   14,817   2,589     70.4
Clothing       120,900    18.8   10,075   1,873     79.5
Groceries      205,900    31.9   17,158   1,619     36.8
Furniture       90,100    14.0    7,508   1,174     62.5
Toys            50,000     7.8    4,167   1,331    151.6


# Quick Reference

### Fundamentals
| Task | Code |
|---|---|
| Load CSV (tolerant / strict) | `np.genfromtxt(f, delimiter=',', skip_header=1)` / `np.loadtxt(f, delimiter=',', skiprows=1)` |
| Save CSV / binary | `np.savetxt(f, a, delimiter=',', fmt='%d')` / `np.save('a.npy', a)` |
| Create | `np.array`, `zeros`, `ones`, `full`, `arange`, `linspace`, `eye`, `default_rng` |
| Inspect | `ndim`, `shape`, `size`, `len()`, `dtype`, `itemsize`, `nbytes` |
| Convert type | `a.astype(int)` |

### Indexing & slicing
| Task | Code |
|---|---|
| Element / row / column | `a[r, c]` / `a[r]` / `a[:, c]` |
| Slice with step | `a[start:stop:step]`, reverse `a[::-1]` |
| Filter | `a[a > x]`, combine with `&`, `\|`, `~` |
| Conditional values | `np.where(cond, x, y)` |
| Pick rows/cols by list | `a[[0, 5]]`, `a[:, [1, 3]]` |
| Positions | `np.where(cond)`, `np.argwhere`, `np.isin` |
| Safe edit | `a.copy()` |

### Transformation
| Task | Code |
|---|---|
| Shape | `reshape`, `ravel`, `flatten`, `.T`, `np.newaxis`, `squeeze` |
| Join / split | `concatenate`, `vstack`, `hstack`, `column_stack` / `split`, `hsplit` |
| Add / remove | `np.append`, `np.insert`, `np.delete` |
| Sort | `np.sort`, `argsort`, `[::-1]` |
| Unique | `np.unique(a, return_counts=True)` |
| Element-wise | `round`, `floor`, `ceil`, `clip`, `abs`, `sqrt`, `log`, `maximum` |
| Scale | min-max, z-score, `keepdims=True` for row shares |
| Change | `np.diff` |
| Missing | `np.isnan`, `np.nan_to_num`, fill with `np.nanmean` |

### Statistics
| Task | Code |
|---|---|
| Aggregates | `sum`, `mean`, `np.median`, `min`, `max`, `np.ptp`, `std`, `var` (`axis=0/1`) |
| Positions | `argmin`, `argmax`, `np.unravel_index` |
| Weighted mean | `np.average(a, weights=w)` |
| Percentiles | `np.percentile(a, [25, 50, 75])`, `np.quantile(a, 0.25)` |
| Outliers | IQR rule, `\|z\| > 2` |
| Cumulative | `cumsum`, `cumprod`, `np.maximum.accumulate` |
| Relationship | `np.corrcoef(a, rowvar=False)`, `np.cov` |
| Distribution | `np.histogram`, `np.digitize`, `np.bincount` |
| Ignore NaN | `np.nanmean`, `np.nansum`, `np.nanmax`, `np.nanpercentile` … |
| Logic | `np.any`, `np.all`, `np.count_nonzero` |

### Common mistakes
- No `skip_header=1` → header becomes a `nan` row; wrong `delimiter` → one `nan` per line.
- `a.shape()` → attributes take no brackets.
- `len(a)` counts rows; `a.size` counts every element.
- Slices exclude `stop`; `a[0:3]` is 3 rows.
- `axis=0` gives one result per **column**.
- `and`/`or` don't work on arrays; use `&`/`|` with brackets.
- `argmax` returns an index; on 2-D without `axis` it's a flattened index.
- Basic slices are views; editing them edits the original.
- `astype(int)` truncates instead of rounding.
- `nan == nan` is False; use `np.isnan`.
- `percentile` uses 0–100; `quantile` uses 0–1.
- NumPy `std` uses `ddof=0`; pandas uses `ddof=1`.